In [0]:
# Instalando as bibliotecas necessárias 

%pip install azure-storage-file-datalake azure-identity pandas pyarrow
dbutils.library.restartPython()

In [0]:
# Importa as bibliotecas usadas para autenticação 

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
import pandas as pd
import os

In [0]:
# Localiza e carrega o arquivo .env com as credenciais do projeto

from dotenv import load_dotenv
from pathlib import Path

env_path = next(Path("/Workspace").rglob(".env"), None)

if env_path:
    load_dotenv(env_path)
    print(".env carregado com sucesso.")
else:
    print(".env não encontrado.")

In [0]:
# Criação da credencial para autenticar o acesso ao ADLS

credential = ClientSecretCredential(
    tenant_id=os.getenv("ADLS_TENANT_ID"),
    client_id=os.getenv("ADLS_CLIENT_ID"),
    client_secret=os.getenv("ADLS_CLIENT_SECRET")
)

print("Credencial do Azure criada com sucesso.")

In [0]:
# Cria a conexão com a conta do Azure Data Lake e com o container dos dados

storage_account_name = "internshipdatalake"
container_name = "raw"

service_client = DataLakeServiceClient(
    account_url=f"https://{storage_account_name}.dfs.core.windows.net",
    credential=credential
)

fs_client = service_client.get_file_system_client(container_name)

print("Conexão com o ADLS criada com sucesso.")

In [0]:
# Localiza todos os arquivos ecommerce_clientes no container raw

arquivos_encontrados = []

for arquivo in fs_client.get_paths(recursive=True):
    if not arquivo.is_directory and arquivo.name.endswith("ecommerce_clientes.parquet"):
        arquivos_encontrados.append(arquivo.name)

print(f"Arquivos encontrados: {len(arquivos_encontrados)}")

for arquivo in arquivos_encontrados:
    print(arquivo)

In [0]:
# Lê todos os arquivos encontrados e junta os registros

from io import BytesIO
import pandas as pd

dfs_clientes = []

for caminho in arquivos_encontrados:
    file_client = fs_client.get_file_client(caminho)
    conteudo = file_client.download_file().readall()

    df_temp = pd.read_parquet(BytesIO(conteudo))
    dfs_clientes.append(df_temp)

df_clientes_pd = pd.concat(dfs_clientes, ignore_index=True)

print("Arquivos lidos com sucesso.")
print(f"Linhas: {len(df_clientes_pd)}")
print(f"Colunas: {len(df_clientes_pd.columns)}")

In [0]:
# Converte os dados do Pandas DataFrame para um PySpark DataFrame

df_clientes = spark.createDataFrame(df_clientes_pd)

print("Conversão para PySpark concluída.")
print(f"Linhas: {df_clientes.count()}")
print(f"Colunas: {len(df_clientes.columns)}")